# Module 44 — Uplift Modeling: Two-Model Approach (Sklearn)

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: Scikit-learn, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 13 (Logistic Regression)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Campaign Data](#3-synthetic-campaign-data)
4. [Two-Model Approach](#4-two-model-approach)
5. [Uplift Prediction](#5-uplift-prediction)
6. [Evaluation](#6-evaluation)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why uplift modeling?

Uplift modeling targets **customers who will respond** to treatment:
- **Persuadables**: Will convert only if treated.
- **Sure things**: Will convert regardless.
- **Lost causes**: Won't convert either way.
- **Sleeping dogs**: Will convert if NOT treated.

**Applications**:
1. **Campaign targeting**: Send offers only to persuadables.
2. **Churn prevention**: Retain at-risk customers.
3. **Upselling**: Target customers likely to upgrade.

**Key advantage**:
- **Incremental impact**: Measures true lift from treatment.
- **ROI optimization**: Avoid wasting budget on non-responders.

In this notebook we will:
1. Generate synthetic campaign response data.
2. Build two separate models for treatment and control.
3. Calculate uplift scores for each customer.
4. Evaluate with Qini curve and AUUC.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Campaign Data

In [ ]:
# Generate synthetic campaign response data
N_CUSTOMERS = 20000

# Generate features
data = {
    'customer_id': range(N_CUSTOMERS),
    'age': np.random.normal(40, 12, N_CUSTOMERS).clip(18, 80).astype(int),
    'income': np.random.lognormal(10.5, 0.8, N_CUSTOMERS),
    'tenure_months': np.random.exponential(24, N_CUSTOMERS).astype(int),
    'previous_purchases': np.random.poisson(5, N_CUSTOMERS),
    'avg_order_value': np.random.exponential(75, N_CUSTOMERS),
    'days_since_last_purchase': np.random.exponential(30, N_CUSTOMERS).astype(int),
}

df = pd.DataFrame(data)

# Treatment assignment (50/50 split)
df['treatment'] = np.random.binomial(1, 0.5, N_CUSTOMERS)

# Generate response with heterogeneous treatment effect
base_response = 0.1
treatment_effect = (
    0.15 * (df['previous_purchases'] > 3).astype(float) +
    0.10 * (df['days_since_last_purchase'] < 30).astype(float) +
    -0.05 * (df['age'] > 60).astype(float)
)

# Response probability
response_prob = base_response + df['treatment'] * treatment_effect
response_prob = response_prob.clip(0.01, 0.99)

# Generate response
df['converted'] = np.random.binomial(1, response_prob)

print(f"✓ Generated {N_CUSTOMERS:,} customer records")
print(f"  Treatment group: {df['treatment'].sum():,}")
print(f"  Control group: {(1 - df['treatment']).sum():,}")
print(f"  Overall conversion: {df['converted'].mean()*100:.2f}%")
print(f"  Treatment conversion: {df[df['treatment']==1]['converted'].mean()*100:.2f}%")
print(f"  Control conversion: {df[df['treatment']==0]['converted'].mean()*100:.2f}%")

---
## §4 · Two-Model Approach

In [ ]:
# Prepare features
feature_cols = ['age', 'income', 'tenure_months', 'previous_purchases',
                'avg_order_value', 'days_since_last_purchase']

X = df[feature_cols]
y = df['converted']
treatment = df['treatment']

# Split data
X_train, X_test, y_train, y_test, treat_train, treat_test = train_test_split(
    X, y, treatment, test_size=0.3, random_state=42
)

# Model for treatment group
model_treatment = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
model_treatment.fit(X_train[treat_train == 1], y_train[treat_train == 1])

# Model for control group
model_control = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
model_control.fit(X_train[treat_train == 0], y_train[treat_train == 0])

print(f"✓ Two models trained")
print(f"  Treatment model: {len(X_train[treat_train == 1]):,} samples")
print(f"  Control model: {len(X_train[treat_train == 0]):,} samples")

---
## §5 · Uplift Prediction

In [ ]:
# Calculate uplift scores
df_test = X_test.copy()
df_test['p_treatment'] = model_treatment.predict_proba(X_test)[:, 1]
df_test['p_control'] = model_control.predict_proba(X_test)[:, 1]
df_test['uplift'] = df_test['p_treatment'] - df_test['p_control']
df_test['actual'] = y_test
df_test['treatment'] = treat_test

print(f"✓ Uplift scores calculated")
print(f"\nUplift distribution:")
print(df_test['uplift'].describe())
print(f"\nCustomers with positive uplift: {(df_test['uplift'] > 0).sum():,} ({(df_test['uplift'] > 0).mean()*100:.1f}%)")

---
## §6 · Evaluation

In [ ]:
# Calculate Qini curve
def qini_curve(y_true, uplift, treatment):
    """Calculate Qini curve."""
    sorted_idx = np.argsort(-uplift)
    y_true_sorted = y_true.values[sorted_idx]
    treatment_sorted = treatment.values[sorted_idx]
    
    n = len(y_true)
    n_treatment = treatment_sorted.sum()
    n_control = n - n_treatment
    
    cumulative_treatment = np.cumsum(y_true_sorted * treatment_sorted)
    cumulative_control = np.cumsum(y_true_sorted * (1 - treatment_sorted))
    
    qini = cumulative_treatment - cumulative_control * (n_treatment / n_control)
    
    return qini / n_treatment

# Calculate Qini
qini = qini_curve(df_test['actual'], df_test['uplift'], df_test['treatment'])
auuc = np.trapz(qini, np.linspace(0, 1, len(qini)))

print(f"✓ Qini analysis complete")
print(f"  AUUC (Area Under Uplift Curve): {auuc:.4f}")
print(f"  Final Qini: {qini[-1]:.4f}")

---
## §7 · Visualization

In [ ]:
# Visualize uplift model
fig = make_subplots(rows=1, cols=2, subplot_titles=['Uplift Distribution', 'Qini Curve'])

fig.add_trace(go.Histogram(
    x=df_test['uplift'],
    nbinsx=50,
    name='Uplift Score'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=np.linspace(0, 1, len(qini)),
    y=qini,
    mode='lines',
    name='Qini Curve'
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Positive uplift = customer worth targeting")
print("   - Negative uplift = avoid targeting (sleeping dogs)")
print("   - Qini curve shows incremental gains from targeting")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Campaign Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Campaign targeting, churn prevention, upselling |
> | **Method** | Two-model approach (treatment vs control models) |
> | **Targeting** | Only target positive uplift customers |
> | **Evaluation** | Qini curve, AUUC, incremental ROI |
> | **Deployment** | Score all customers, target top uplift deciles |
>
> ### 💡 Targeting Strategy
>
> | Uplift Segment | Description | Action |
> |----------------|-------------|--------|
> | High positive | Persuadables | Target aggressively |
> | Low positive | Mild responders | Target if budget allows |
> | Near zero | Sure things/lost causes | Don't target |
> | Negative | Sleeping dogs | Avoid targeting |
>
> ### 🔑 Key Takeaways
>
> 1. **Uplift modeling targets persuadables**, not just likely converters.
> 2. **Two-model approach** is simple and effective.
> 3. **Qini curve** measures incremental impact.
> 4. **Avoid sleeping dogs** - customers who would churn if targeted.
> 5. **Uplift targeting** improves campaign ROI by 20-40%.